# Tab 1 - Silhouette analysis

**Expected directory layout** (mirrors `enhanced_simulation_main.R`):
```
output/
  tab1/
    baseline/
      sim_seed*_c*_k*_gammapar*_frailty*_censor*.Rds
```

In [57]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import rdata      
from pathlib import Path
from sklearn.metrics import adjusted_rand_score

# ── Configuration ──────────────────────────────────────────────────────────
BASE_DIR  = Path("output/tab1")   # root output folder from enhanced_simulation_main.R
SCENARIO  = "baseline"            # which sub-folder / scenario to analyse
N_CLUSTERS_EXPECTED = 3           # keep only runs where n_components == 3

folder = BASE_DIR / SCENARIO
rds_files = sorted(folder.glob("*.Rds"))
print(f"Found {len(rds_files)} .Rds files in '{folder}'")

Found 800 .Rds files in 'output/tab1/baseline'


In [58]:
# ── Load & process all .Rds files ─────────────────────────────────────────

records = []
skipped = 0

for file in rds_files:
    fname = file.name

    # ── Determine censoring from filename ──────────────────────────────
    if "censoradministrative" in fname:
        censor_name = "Administrative"
    elif "censornormal" in fname:
        censor_name = "Normal"
    else:
        skipped += 1
        continue

    # ── Read the Rds file ──────────────────────────────────────────────
    try:
        obj = rdata.read_rds(file)
        obj = {str(k): v for k, v in obj.items()}

    except Exception as e:
        print(f"  [WARNING] Could not read {fname}: {e}")
        skipped += 1
        continue

    record = {
        "seed": float(obj["seed"][0]),
        "c": float(obj["c"][0]),
        "k": float(obj["k"][0]),
        "gammapar": float(obj["gammapar"][0]),
        "n_components": float(obj["n_components"][0]),
        "S": obj["S"],
        "loglik": float(obj["loglik"][0]),
        "iterations": float(obj["iterations"][0]),
        "lambdaoptim": float(obj["lambdaoptim"][0]),
        "censor": censor_name,
        "silhouette": float(obj["silhouette"]['si.summary.Mean'][0])
    }

    # expand the estimates into individual columns
    for i, val in enumerate(obj["estimate"]):
        record[f"estimate_{i}"] = float(val)

    records.append(record)

print(f"Processed {len(rds_files) - skipped} files  |  skipped {skipped}")

# build dataframe
df_pen = pd.DataFrame(records)

### now we transform back the estimate_2 which is $\xi^\rho$ to make \xi
df_pen['estimate_2'] = df_pen['estimate_2']**(1/df_pen['estimate_1'])

df_pen


Processed 800 files  |  skipped 0


,seed,c,k,gammapar,n_components,S,loglik,iterations,lambdaoptim,censor,silhouette,estimate_0,estimate_1,estimate_2,estimate_3
0,0.0,2.0,20.0,0.0001,2.0,"[[0.0, 0.0, 0.0, 0.0, 0.09633745720234778, 0.0...",-793.950589,3.0,694.444444,Administrative,0.682282,1.060280,2.527421,0.009982,0.704208
1,0.0,2.0,20.0,0.0001,2.0,"[[0.0, 0.0, 0.0, 0.0, 0.2322726331895355, 0.0,...",-935.783889,41.0,1274.619647,Normal,0.603982,1.034381,2.475448,0.010109,0.686214
2,0.0,3.0,20.0,0.0001,3.0,"[[0.0, 0.0, 0.0, 0.0, 0.09537755357401805, 0.0...",-793.949225,3.0,1000.000000,Administrative,0.840600,1.061340,2.526761,0.009975,0.704051
3,0.0,3.0,20.0,0.0001,3.0,"[[0.0, 0.0, 0.0, 0.0, 0.21143434653261212, 0.0...",-935.685750,3.0,1440.000000,Normal,0.797117,1.034331,2.476860,0.010112,0.686636
4,0.0,4.0,20.0,0.0001,4.0,"[[0.0, 0.0, 0.0, 0.0, 0.09537755357401805, 0.0...",-793.948962,25.0,83.359641,Administrative,0.715008,1.061340,2.526761,0.009975,0.704051
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,9.0,3.0,20.0,0.0001,3.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",-926.522725,3.0,1440.000000,Normal,0.782992,0.620930,2.584001,0.009796,0.723861
796,9.0,4.0,20.0,0.0001,4.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",-818.244466,21.0,42.941481,Administrative,0.723313,0.612729,2.550577,0.009647,0.717770
797,9.0,4.0,20.0,0.0001,4.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",-926.523759,37.0,1078.758401,Normal,0.676511,0.620604,2.584172,0.009795,0.723932
798,9.0,5.0,20.0,0.0001,5.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",-818.244376,18.0,38.943283,Administrative,0.607993,0.612729,2.550577,0.009647,0.717770


In [59]:
df_pen[df_pen.k == 20].groupby(['k'])['iterations'].value_counts()

k     iterations
20.0  3.0           262
      200.0          41
      31.0           32
      32.0           30
      30.0           25
      29.0           23
      33.0           22
      38.0           20
      26.0           20
      35.0           19
      37.0           19
      21.0           19
      34.0           19
      28.0           18
      36.0           17
      24.0           16
      39.0           15
      25.0           15
      23.0           15
      40.0           14
      27.0           14
      22.0           13
      18.0           12
      4.0            12
      20.0           12
      41.0           11
      16.0           10
      44.0            7
      14.0            6
      19.0            6
      15.0            6
      17.0            5
      11.0            4
      46.0            3
      45.0            3
      42.0            3
      43.0            2
      64.0            2
      48.0            2
      74.0            1
      8.0             1

In [60]:
df = df_pen

In [61]:
df.n_components.value_counts()

n_components
3.0    241
5.0    200
4.0    199
2.0    160
Name: count, dtype: int64

In [62]:
df.loc[df.gammapar!=0, 'n_components'].value_counts()

n_components
3.0    241
5.0    200
4.0    199
2.0    160
Name: count, dtype: int64

# select the censoring type here: either "Normal" or "Administrative"

In [63]:
df = df.loc[(df['censor'] == 'Administrative')]

In [64]:
df2 = df.loc[(df['c'] == 2) & (df['n_components'] == 2)]
df2

,seed,c,k,gammapar,n_components,S,loglik,iterations,lambdaoptim,censor,silhouette,estimate_0,estimate_1,estimate_2,estimate_3
0,0.0,2.0,20.0,0.0001,2.0,"[[0.0, 0.0, 0.0, 0.0, 0.09633745720234778, 0.0...",-793.950589,3.0,694.444444,Administrative,0.682282,1.060280,2.527421,0.009982,0.704208
8,10.0,2.0,20.0,0.0001,2.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2927816...",-886.886635,3.0,694.444444,Administrative,0.660967,0.613794,2.462098,0.009932,0.683033
24,12.0,2.0,20.0,0.0001,2.0,"[[0.0, 0.3807064575076612, 0.0, 0.0, 0.0, 0.0,...",-773.215099,3.0,694.444444,Administrative,0.669950,0.012806,2.554392,0.008568,0.721661
32,13.0,2.0,20.0,0.0001,2.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",-746.738400,3.0,694.444444,Administrative,0.674328,0.257934,2.400484,0.009629,0.685442
40,14.0,2.0,20.0,0.0001,2.0,"[[0.0, 0.8085454896567894, 0.0, 0.0, 0.0, 0.0,...",-894.176283,3.0,694.444444,Administrative,0.663503,0.290453,2.493769,0.009602,0.682743
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
736,93.0,2.0,20.0,0.0001,2.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.16...",-893.636251,3.0,694.444444,Administrative,0.669644,0.426754,2.384884,0.010881,0.653931
744,94.0,2.0,20.0,0.0001,2.0,"[[0.0, 0.0, 0.6757408615412923, 0.0, 0.0, 0.0,...",-838.130784,4.0,1.147200,Administrative,0.684467,0.231811,2.468788,0.010719,0.678785
752,95.0,2.0,20.0,0.0001,2.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",-788.525554,3.0,694.444444,Administrative,0.671727,0.921781,2.501155,0.009721,0.705831
760,96.0,2.0,20.0,0.0001,2.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5012118...",-868.469748,3.0,694.444444,Administrative,0.666324,0.434053,2.397070,0.009548,0.668567


In [65]:
df3 = df.loc[(df['c'] == 3) & (df['n_components'] == 3)]
df3

,seed,c,k,gammapar,n_components,S,loglik,iterations,lambdaoptim,censor,silhouette,estimate_0,estimate_1,estimate_2,estimate_3
2,0.0,3.0,20.0,0.0001,3.0,"[[0.0, 0.0, 0.0, 0.0, 0.09537755357401805, 0.0...",-793.949225,3.0,1000.0,Administrative,0.840600,1.061340,2.526761,0.009975,0.704051
10,10.0,3.0,20.0,0.0001,3.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2712702...",-886.879626,3.0,1000.0,Administrative,0.828825,0.614014,2.461984,0.009928,0.682982
18,11.0,3.0,20.0,0.0001,3.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",-842.692679,3.0,1000.0,Administrative,0.839108,0.711825,2.461433,0.010130,0.693058
26,12.0,3.0,20.0,0.0001,3.0,"[[0.0, 0.37764339200724956, 0.0, 0.0, 0.0, 0.0...",-773.215400,3.0,1000.0,Administrative,0.845073,0.012872,2.554454,0.008567,0.721686
34,13.0,3.0,20.0,0.0001,3.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",-746.738190,3.0,1000.0,Administrative,0.835996,0.257934,2.400484,0.009629,0.685442
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
762,96.0,3.0,20.0,0.0001,3.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5011934...",-868.468830,3.0,1440.0,Administrative,0.827010,0.434041,2.397371,0.009549,0.668632
770,97.0,3.0,20.0,0.0001,3.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5123237...",-806.542686,3.0,1000.0,Administrative,0.843887,0.095965,2.406505,0.009564,0.677850
778,98.0,3.0,20.0,0.0001,3.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",-771.948765,3.0,1000.0,Administrative,0.839904,0.205789,2.489559,0.010175,0.696764
786,99.0,3.0,20.0,0.0001,3.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",-915.804804,3.0,1000.0,Administrative,0.824973,0.824493,2.505397,0.009921,0.690577


In [66]:
df4 = df.loc[(df['c'] == 4) & (df['n_components'] == 4)]
df4

,seed,c,k,gammapar,n_components,S,loglik,iterations,lambdaoptim,censor,silhouette,estimate_0,estimate_1,estimate_2,estimate_3
4,0.0,4.0,20.0,0.0001,4.0,"[[0.0, 0.0, 0.0, 0.0, 0.09537755357401805, 0.0...",-793.948962,25.0,83.359641,Administrative,0.715008,1.061340,2.526761,0.009975,0.704051
12,10.0,4.0,20.0,0.0001,4.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2711850...",-886.878273,36.0,498.311133,Administrative,0.722144,0.614014,2.461984,0.009928,0.682982
20,11.0,4.0,20.0,0.0001,4.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",-842.692196,15.0,13.175636,Administrative,0.728122,0.711825,2.461433,0.010130,0.693058
28,12.0,4.0,20.0,0.0001,4.0,"[[0.0, 0.3776852342386426, 0.0, 0.0, 0.0, 0.0,...",-773.214728,25.0,87.974687,Administrative,0.730047,0.012806,2.554392,0.008568,0.721661
36,13.0,4.0,20.0,0.0001,4.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",-746.738071,33.0,328.958714,Administrative,0.717639,0.257934,2.400484,0.009629,0.685442
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
764,96.0,4.0,20.0,0.0001,4.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5012118...",-868.469761,30.0,176.780468,Administrative,0.467365,0.434053,2.397070,0.009548,0.668567
772,97.0,4.0,20.0,0.0001,4.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5277488...",-806.543101,22.0,49.951583,Administrative,0.725180,0.095964,2.406765,0.009565,0.677836
780,98.0,4.0,20.0,0.0001,4.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",-771.948762,18.0,22.399901,Administrative,0.696665,0.205962,2.489622,0.010176,0.696732
788,99.0,4.0,20.0,0.0001,4.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",-915.804737,18.0,25.233609,Administrative,0.688061,0.824493,2.505397,0.009921,0.690577


In [67]:
df5 = df.loc[(df['c'] == 5) & (df['n_components'] == 5)]
df5

,seed,c,k,gammapar,n_components,S,loglik,iterations,lambdaoptim,censor,silhouette,estimate_0,estimate_1,estimate_2,estimate_3
6,0.0,5.0,20.0,0.0001,5.0,"[[0.0, 0.0, 0.0, 0.0, 0.12287969637288092, 0.0...",-793.948920,22.0,79.436508,Administrative,0.623981,1.061340,2.526761,0.009975,0.704051
14,10.0,5.0,20.0,0.0001,5.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2927816...",-886.885259,22.0,80.390771,Administrative,0.333569,0.613794,2.462098,0.009932,0.683033
22,11.0,5.0,20.0,0.0001,5.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",-842.692374,16.0,32.949472,Administrative,0.622184,0.711825,2.461433,0.010130,0.693058
30,12.0,5.0,20.0,0.0001,5.0,"[[0.0, 0.3776852342386426, 0.0, 0.0, 0.0, 0.0,...",-773.214880,21.0,86.584609,Administrative,0.614737,0.012806,2.554392,0.008568,0.721661
38,13.0,5.0,20.0,0.0001,5.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",-746.738493,30.0,306.249985,Administrative,0.629737,0.257934,2.400484,0.009629,0.685442
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
766,96.0,5.0,20.0,0.0001,5.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5012118...",-868.469820,28.0,194.738206,Administrative,0.480818,0.434053,2.397070,0.009548,0.668567
774,97.0,5.0,20.0,0.0001,5.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5268857...",-806.543612,19.0,50.311361,Administrative,0.608771,0.095964,2.406765,0.009565,0.677836
782,98.0,5.0,20.0,0.0001,5.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",-771.948624,15.0,21.458827,Administrative,0.607921,0.205962,2.489622,0.010176,0.696732
790,99.0,5.0,20.0,0.0001,5.0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",-915.805778,24.0,116.997963,Administrative,0.611918,0.824493,2.505397,0.009921,0.690577


In [68]:
for df in [df2, df3, df4, df5]:
    print('Mean:')
    print(round(df['silhouette'].mean(),3))
    print('Median:')
    print(round(df['silhouette'].median(),3))
    print('SD:')
    print(round(df['silhouette'].std(),3))
    print('n:')
    print(round(df['silhouette'].count(),3))
    print('\n')

Mean:
0.66
Median:
0.67
SD:
0.046
n:
67


Mean:
0.792
Median:
0.832
SD:
0.116
n:
100


Mean:
0.661
Median:
0.718
SD:
0.121
n:
100


Mean:
0.567
Median:
0.612
SD:
0.113
n:
100


